# Leakage-safe feature engineering
The important rule is not ‘encode categories somehow’, but ‘fit every data-dependent transform on training folds only’. A `Pipeline` + `ColumnTransformer` makes that contract explicit.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score

rng=np.random.default_rng(42); n=1200
df=pd.DataFrame({'age':rng.normal(40,12,n),'income':rng.lognormal(10.5,.5,n),'city':rng.choice(['A','B','C'],n,p=[.5,.3,.2]),'tenure':rng.exponential(4,n)})
df.loc[rng.choice(n,80,replace=False),'income']=np.nan
logit=-2 + .035*(df.age-40) + .45*(df.city=='C') - .18*df.tenure + .000015*df.income.fillna(df.income.median())
y=rng.binomial(1,1/(1+np.exp(-logit)))
num=['age','income','tenure']; cat=['city']
pre=ColumnTransformer([('num',make_pipeline(SimpleImputer(strategy='median'),StandardScaler()),num),('cat',OneHotEncoder(handle_unknown='ignore'),cat)])
model=make_pipeline(pre,LogisticRegression(max_iter=3000))
cv=StratifiedKFold(5,shuffle=True,random_state=42)
cross_val_score(model,df,y,cv=cv,scoring='roc_auc').mean().round(4)


## Takeaway
The feature representation is part of the model. If imputation, scaling or category vocabularies are learned before cross-validation, validation is contaminated even if the final estimator never directly sees the test labels.